# 01A｜向量、点与坐标表示：先分清“物体”和“描述物体的数字”

**定位：教材级基础微课（v0.12）｜预计 90–120 分钟｜Mac CPU 即可**

对应[第 01 章](../course/01-coordinates-and-kinematics.md)和
[教材级 Notebook 标准](../course/FOUNDATION_NOTEBOOK_STANDARD.md)。

这一本只解决一个根问题：**空间中的几何对象，与我们用某个坐标系写出的数字，
不是同一件事。** 如果这一步含糊，后续的齐次变换、相机外参、Panda 末端控制和
失败位置分析都会变成“记矩阵顺序”。

前置知识：Python 函数、NumPy 一维数组、矩阵乘法。这里不假定机器人学知识。
第一遍请逐格运行；完成全部练习后再用 `Restart Kernel and Run All Cells` 验证。

## 学习目标

完成本节后，你应该能够：

1. 严格区分几何点、自由向量和它们的坐标数组；
2. 用“原点 + 一组基向量”定义一个三维坐标系；
3. 解释记号 ${}^A p$ 中上标 A 的语义；
4. 从坐标系定义推导点和向量的坐标转换公式；
5. 在代码中始终标注 frame、单位和 shape；
6. 解释为什么相机图像的“左”不能直接等同于 world 负 y。

## 本节知识地图

```text
几何对象
├── 点 p：表示位置
└── 自由向量 v：表示方向和长度
       ↓ 选择坐标系 F=(原点, 三根基轴)
坐标表示  ^F p, ^F v ∈ R³
       ↓ 改变坐标系
同一个对象，不同的三个数
       ↓ 项目迁移
cube world position / tip displacement / camera pixels
```

## 开始前诊断

先不要向下阅读定义。根据当前理解选择答案；三个控件默认都为空，因此不会提前
暗示正确选项。选完后先保留，不查资料。学完定义后我们会回到这些答案。

1. 坐标系原点平移时，空间中的自由向量会不会改变？
2. 两个数组都是 `[1, 2, 3]`，是否足以判断它们表示同一个空间位置？
3. 两个几何点可以像普通向量那样相加吗？

In [ ]:
from pathlib import Path
import sys

# 从当前目录向上寻找仓库根目录；不要依赖 VS Code 或 Jupyter 的启动位置。
ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file()
)
sys.path.insert(0, str(ROOT / "docs" / "notebooks"))

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from course_feedback import check_choice, check_value, save_progress
from course_utils import assert_course_kernel

# 若项目已有 .venv，这个检查可以防止 VS Code 误选系统 Python。
assert_course_kernel(ROOT)
print("Python kernel:", sys.executable)

vector_translation_quiz = widgets.RadioButtons(
    options=[("请选择", None), ("会改变", "changes"), ("不会改变", "unchanged")],
    value=None,
    description="自由向量",
)
array_identity_quiz = widgets.RadioButtons(
    options=[("请选择", None), ("足以", "enough"), ("不够", "insufficient")],
    value=None,
    description="相同数组",
)
point_addition_quiz = widgets.RadioButtons(
    options=[("请选择", None), ("总是有意义", "valid"), ("一般没有定义", "undefined")],
    value=None,
    description="点 + 点",
)
display(vector_translation_quiz, array_identity_quiz, point_addition_quiz)

## 严格定义

### 定义 1：几何点

几何点 $p$ 是仿射空间中的一个位置。点本身不是“从原点出发的箭头”，因此在
尚未选择坐标系时，不能把它等同于 $mathbb{R}^3$ 中的三元数组。

选择坐标系 F 后，点 $p$ 才获得坐标表示：

$$
{}^F p=
\begin{bmatrix}p_x\\p_y\\p_z\end{bmatrix}\in\mathbb{R}^3.
$$

${}^F p$ 读作“点 $p$ 在 F 坐标系中的坐标”。上标 F 属于坐标描述，不是指数。

### 定义 2：自由向量

自由向量 $v$ 表示方向和长度，不绑定某个起点。选择坐标系 F 后，它的坐标为
${}^F v\in\mathbb{R}^3$。两个点的差 $q-p$ 是向量；点加向量 $p+v$ 是点；
但“点 + 点”在普通仿射几何中没有坐标系无关的定义。

### 定义 3：右手正交坐标系

三维坐标系 F 由以下对象组成：

$$
F=(o_F, e^F_x,e^F_y,e^F_z),
$$

其中 $o_F$ 是原点，三根基向量长度为 1、两两正交，并满足右手规则
$e^F_x\times e^F_y=e^F_z$。

### 定义 4：旋转矩阵

将 B 系三根基轴用 A 系坐标写成列向量：

$$
{}^A R_B=
\begin{bmatrix}
|&|&|\\
{}^A e^B_x&{}^A e^B_y&{}^A e^B_z\\
|&|&|
\end{bmatrix}.
$$

合法三维旋转矩阵属于特殊正交群：

$$
SO(3)=\left\{R\in\mathbb{R}^{3\times3}\mid R^TR=I,\ \det(R)=1\right\}.
$$

$R^TR=I$ 保证基轴单位正交；$\det(R)=1$ 排除镜像反射。

## 关键概念与符号

## 符号、单位与 shape

| 对象 | 数学记号 | NumPy shape | 典型单位 | 必须注明 |
| --- | --- | ---: | --- | --- |
| 点的坐标 | ${}^F p$ | `(3,)` | m | 表达坐标系 F |
| 自由向量坐标 | ${}^F v$ | `(3,)` | 依物理量而定 | 表达坐标系 F |
| 坐标系原点 | ${}^A o_B$ | `(3,)` | m | B 原点用 A 表达 |
| 旋转矩阵 | ${}^A R_B$ | `(3, 3)` | 无量纲 | 输入 B、输出 A |
| 像素坐标 | $(u,v)$ | `(2,)` | pixel | 不是三维 world 坐标 |

**同样的 shape 不代表同样的类型。** `cube_position_world`、
`tip_displacement_world` 和 `rgb_pixel` 都可能由数组保存，但它们不能随意相加。

本课程采用列向量约定：矩阵写在坐标向量左侧。变量名 `a_from_b` 与
${}^A R_B$ 同义，表示“输入 B 表达，输出 A 表达”。

## 直观解释

把坐标系想成“原点 + 三把带方向的尺子”是有帮助的：换一套尺子不会移动方块，
只会改变描述方块的三个数字。但类比有边界：真实坐标转换不是把标签重命名，
而是由基向量的内积严格确定。

为什么要区分点和向量？假设整个实验桌向右移动 1 m：

- 方块的位置坐标会随原点变化；
- “夹爪向上移动 5 mm”这个方向与长度不会因原点平移而改变。

代码把二者都存成 `(3,)`，因此 `point_world`、`delta_tip_world` 之类的名字不是
风格偏好，而是在缺少静态几何类型时保护语义。

## 原理与推导

令 $E_F=[e^F_x\ e^F_y\ e^F_z]$，即把 F 的三根基轴放在矩阵列中。
点 $p$ 的几何位置可以写成：

$$
p=o_F+E_F{}^F p. \tag{1}
$$

对向量 $v$，没有原点项：

$$
v=E_F{}^F v. \tag{2}
$$

现在让 A、B 两个坐标系描述同一点。分别代入式 (1)：

$$
p=o_A+E_A{}^A p=o_B+E_B{}^B p.
$$

两边减去 $o_A$，再左乘 $E_A^T$。因为 A 的基正交，$E_A^TE_A=I$：

$$
{}^A p=E_A^T(o_B-o_A)+E_A^TE_B{}^B p.
$$

定义
${}^A t_B=E_A^T(o_B-o_A)$、${}^A R_B=E_A^TE_B$，得到：

$$
{}^A p={}^A R_B{}^B p+{}^A t_B. \tag{3}
$$

对自由向量使用式 (2)，原点相减项不存在：

$$
{}^A v={}^A R_B{}^B v. \tag{4}
$$

式 (3) 和式 (4) 的唯一区别就是平移项。这不是人为规定，而是从点与向量的定义
推导出来的。

## 先预测

A 为 world。B 原点在 A 中是 `[2, 1, 0] m`，B 相对 A 绕 z 轴逆时针旋转
90°。在运行代码前纸笔完成：

1. B 的 x 轴用 A 表达是什么？
2. B 中点 `[1, 0, 0] m` 用 A 表达是什么？
3. B 中自由向量 `[1, 0, 0] m` 用 A 表达是什么？
4. 为什么第 2、3 题的答案相差 `[2, 1, 0]`？

## Worked example

90° 旋转时，B 的 x 轴在 A 中指向 `[0, 1, 0]`，B 的 y 轴在 A 中指向
`[-1, 0, 0]`，所以：

$$
{}^A R_B=
\begin{bmatrix}
0&-1&0\\
1& 0&0\\
0& 0&1
\end{bmatrix}.
$$

对点：

$$
{}^A p={}^A R_B[1,0,0]^T+[2,1,0]^T=[2,2,0]^T.
$$

对自由向量：

$$
{}^A v={}^A R_B[1,0,0]^T=[0,1,0]^T.
$$

两个结果都正确，却回答不同问题：前者是位置，后者是方向和长度。

## 运行与观察

下面实现式 (3) 和式 (4)。先读注释，再运行。特别留意函数名和变量名如何携带
`point/vector` 与 `world/from_b` 语义。

In [ ]:
def rotation_z_deg(angle_deg: float) -> np.ndarray:
  '''Returns world_from_local rotation for a z-axis angle, shape (3, 3).'''
  # NumPy 的 sin/cos 使用弧度；接口接收角度是为了让本节手算更直观。
  angle_rad = np.deg2rad(angle_deg)
  cosine, sine = np.cos(angle_rad), np.sin(angle_rad)
  return np.array([
      [cosine, -sine, 0.0],
      [sine, cosine, 0.0],
      [0.0, 0.0, 1.0],
  ])


def express_point_in_world(
    point_b_m: np.ndarray,
    world_from_b_rotation: np.ndarray,
    b_origin_world_m: np.ndarray,
) -> np.ndarray:
  '''Maps one point from B coordinates to world coordinates, all shapes explicit.'''
  # point_b_m: 点 p 在 B 中的坐标，shape=(3,), unit=m。
  # world_from_b_rotation: B 基轴在 world 中的列表示，shape=(3, 3)。
  # b_origin_world_m: B 原点在 world 中的位置，shape=(3,), unit=m。
  assert point_b_m.shape == (3,)
  assert world_from_b_rotation.shape == (3, 3)
  assert b_origin_world_m.shape == (3,)
  return world_from_b_rotation @ point_b_m + b_origin_world_m


def express_vector_in_world(
    vector_b: np.ndarray,
    world_from_b_rotation: np.ndarray,
) -> np.ndarray:
  '''Changes a free vector's coordinate basis; no origin translation is allowed.'''
  assert vector_b.shape == (3,)
  return world_from_b_rotation @ vector_b


world_from_b_rotation = rotation_z_deg(90.0)
b_origin_world_m = np.array([2.0, 1.0, 0.0])
point_b_m = np.array([1.0, 0.0, 0.0])
direction_b = np.array([1.0, 0.0, 0.0])

point_world_m = express_point_in_world(
    point_b_m, world_from_b_rotation, b_origin_world_m
)
direction_world = express_vector_in_world(direction_b, world_from_b_rotation)

print("point in world [m]:", np.round(point_world_m, 6))
print("direction in world:", np.round(direction_world, 6))

In [ ]:
# 图中所有坐标都已经用 world 表达，因此可以画在同一组坐标轴上。
fig, axis = plt.subplots(figsize=(7, 6))
axis.axhline(0.0, color="0.85")
axis.axvline(0.0, color="0.85")

# world 的两根单位轴从 world 原点出发。
axis.quiver(0, 0, 1, 0, angles="xy", scale_units="xy", scale=1, color="tab:red")
axis.quiver(0, 0, 0, 1, angles="xy", scale_units="xy", scale=1, color="tab:green")

# B 的基轴必须从 B 原点出发，但箭头方向取旋转矩阵的两列。
b_x_world = world_from_b_rotation[:2, 0]
b_y_world = world_from_b_rotation[:2, 1]
axis.quiver(*b_origin_world_m[:2], *b_x_world, angles="xy", scale_units="xy", scale=1)
axis.quiver(*b_origin_world_m[:2], *b_y_world, angles="xy", scale_units="xy", scale=1)
axis.scatter(*point_world_m[:2], s=90, label="physical point p")

axis.set(
    xlim=(-0.5, 3.5),
    ylim=(-0.5, 3.5),
    aspect="equal",
    xlabel="world x [m]",
    ylabel="world y [m]",
    title="One physical point described through frame B",
)
axis.legend()
plt.show()

## 故意出错

下面把自由向量误用点公式处理。它不会产生异常，shape 也是 `(3,)`，甚至数值看上去
很合理，因此属于机器人代码中危险的**语义错误**。运行后解释：为什么原点从
`[2,1,0]` 变成 `[20,10,0]` 时，“向右一米”的方向不应该跟着改变？

In [ ]:
wrong_direction_world = express_point_in_world(
    direction_b,
    world_from_b_rotation,
    b_origin_world_m,
)
check_value(
    "vector must ignore translation",
    wrong_direction_world,
    direction_world,
    hint="a free vector has no origin; use only world_from_b_rotation @ direction_b",
)

## 动手修改

先预测再操作：保持 B 中点 `[1,0,0]` 不变，改变 B 原点和角度。

- 改原点时，点坐标怎样变化？方向怎样变化？
- 改角度时，点和方向的哪一部分一起变化？

移动控件后重新运行下一格。控件只是输入界面，真正的概念仍是式 (3)、(4)。

In [ ]:
angle_widget = widgets.SelectionSlider(
    options=[0.0, 45.0, 90.0, 180.0], value=90.0, description="angle°"
)
origin_x_widget = widgets.FloatSlider(
    value=2.0, min=-1.0, max=3.0, step=0.5, description="origin x"
)
display(angle_widget, origin_x_widget)

# 重新运行本格时才会读取控件当前值。
trial_world_from_b = rotation_z_deg(float(angle_widget.value))
trial_origin_world_m = np.array([origin_x_widget.value, 1.0, 0.0])
trial_point_world_m = express_point_in_world(
    point_b_m, trial_world_from_b, trial_origin_world_m
)
trial_direction_world = express_vector_in_world(direction_b, trial_world_from_b)
print("point [m]:", np.round(trial_point_world_m, 4))
print("direction:", np.round(trial_direction_world, 4))

## 分层练习

**Level 1｜模仿。** B 不旋转，原点为 `[3,-1,0] m`。手算 B 中点
`[0.5,0.5,0] m` 的 world 坐标，再调用 `express_point_in_world` 验证。

**Level 2｜补全。** 根据式 (3) 反解：已知 ${}^A p$、${}^A R_B$ 和
${}^A t_B$，写函数求 ${}^B p$。提示：左乘旋转转置，但必须先减平移。

**Level 3｜迁移。** 假设 `cube_position_world=[0.5,-0.03,0.02] m`，相机像素
中方块位于图像左侧。写两句话说明为什么这两条信息不能直接互换，并列出还需要的
相机内参、外参和投影模型。

建议把答案写进 `notes/01a-coordinate-representations.md`，不要直接修改发布 Notebook。

In [ ]:
def express_world_point_in_b(
    point_world_m: np.ndarray,
    world_from_b_rotation: np.ndarray,
    b_origin_world_m: np.ndarray,
) -> np.ndarray:
  '''Inverse of equation (3): world point coordinates -> B coordinates.'''
  # 先减去 B 原点，把“位置”变成相对 B 原点的位移向量。
  displacement_world_m = point_world_m - b_origin_world_m
  # SO(3) 的逆等于转置：B_from_world = world_from_b.T。
  return world_from_b_rotation.T @ displacement_world_m


recovered_point_b_m = express_world_point_in_b(
    point_world_m, world_from_b_rotation, b_origin_world_m
)
print("recovered point in B [m]:", np.round(recovered_point_b_m, 6))

## 回看开始诊断

现在读取开头三个控件。如果仍未选择，反馈只会提示作答，不会替你填入答案。
答错时回到“严格定义”和式 (3)、(4)，不要靠反复猜选项。

In [ ]:
diagnostic_checks = [
    check_choice(
        "原点平移对自由向量",
        vector_translation_quiz.value,
        "unchanged",
        hint="比较点公式 (3) 和向量公式 (4)。",
        explanation="自由向量没有绑定原点，换原点不改变几何向量。",
    ),
    check_choice(
        "相同数组是否足够",
        array_identity_quiz.value,
        "insufficient",
        hint="数组还缺少对象类型、表达 frame 和单位。",
        explanation="相同数字在不同 frame 中可以表示不同位置。",
    ),
    check_choice(
        "点加点",
        point_addition_quiz.value,
        "undefined",
        hint="仿射空间允许点减点、点加向量。",
        explanation="点 + 点通常没有坐标系无关的几何意义。",
    ),
]

## 自测

运行断言前，逐条口头说明它检查的是定义、旋转约束还是往返一致性。如果只能说
“应该等于这个数组”，说明还没有把程序和原理连接起来。

In [ ]:
# SO(3) 不变量：单位正交，且保持右手性。
assert np.allclose(world_from_b_rotation.T @ world_from_b_rotation, np.eye(3))
assert np.isclose(np.linalg.det(world_from_b_rotation), 1.0)

# 手算例子的两个不同几何类型。
assert np.allclose(point_world_m, [2.0, 2.0, 0.0])
assert np.allclose(direction_world, [0.0, 1.0, 0.0])

# 坐标转换往返后，应恢复同一个 B 坐标表示。
assert np.allclose(recovered_point_b_m, point_b_m)

# 两点之差是向量，所以两点共同平移后，它们的差保持不变。
another_point_b_m = np.array([0.0, 2.0, 0.0])
another_point_world_m = express_point_in_world(
    another_point_b_m, world_from_b_rotation, b_origin_world_m
)
assert np.allclose(
    another_point_world_m - point_world_m,
    world_from_b_rotation @ (another_point_b_m - point_b_m),
)
print("PASS: object semantics, SO(3), and coordinate round trip")

## 项目源码连接

在 Panda 环境中：

- `data.xpos[self._obj_body]` 是方块位置，语义是 world 中的点，单位 m；
- `increment`/`scaled_pos` 是末端位移向量，不是绝对位置；
- `new_tip_pos = current_tip_pos + scaled_pos` 是“点 + 向量 = 点”；
- 相机 RGB 是投影后的像素数组，不能与 `box_pos` 直接比较。

下一格从真实文件抽取相关语句。教学代码使用 NumPy；正式环境使用 JAX/MJX 数组，
但点和向量的几何语义不因数组库改变。

In [ ]:
source_path = (
    ROOT
    / "mujoco_playground/_src/manipulation/franka_emika_panda/pick_cartesian.py"
)
source_lines = source_path.read_text(encoding="utf-8").splitlines()
anchors = ("box_pos = data.xpos", "scaled_pos =", "new_tip_pos =")

print("Source:", source_path.relative_to(ROOT))
for line_number, source_line in enumerate(source_lines, start=1):
  if any(anchor in source_line for anchor in anchors):
    print(f"{line_number:4d}: {source_line.strip()}")

## Exit ticket

先在下面字典中写入自己的短答案，再运行反馈格。不要从反馈代码复制字符串；真正
通过还要求你能口头解释原因。

- `point_minus_point`：结果的几何类型是什么？
- `frame_required`：一个三元数组要成为完整几何描述还缺什么？
- `vector_translation`：自由向量换原点时如何变化？
- `project_operation`：`current_tip_pos + scaled_pos` 属于哪种合法运算？

In [ ]:
# 保持 None，表示尚未作答；请替换为自己的短答案。
exit_answers = {
    "point_minus_point": None,
    "frame_required": None,
    "vector_translation": None,
    "project_operation": None,
}
print("Edit exit_answers, then run the next cell.")

In [ ]:
exit_checks = [
    check_choice(
        "点减点",
        exit_answers["point_minus_point"],
        "vector",
        hint="结果表示从一个位置到另一个位置的方向与长度。",
        explanation="点减点得到自由向量。",
    ),
    check_choice(
        "完整坐标描述",
        exit_answers["frame_required"],
        "frame and unit",
        hint="同一数组可以属于 camera、base 或 world。",
        explanation="至少需要对象类型、表达 frame 和单位；本题要求后两项。",
    ),
    check_choice(
        "自由向量换原点",
        exit_answers["vector_translation"],
        "unchanged",
        hint="自由向量公式没有平移项。",
        explanation="换原点不改变自由向量。",
    ),
    check_choice(
        "Panda 位置更新",
        exit_answers["project_operation"],
        "point plus vector",
        hint="current_tip_pos 是位置，scaled_pos 是增量。",
        explanation="点加位移向量得到新的点。",
    ),
]
exit_ticket_passed = all(exit_checks)

SAVE_PROGRESS = False
if SAVE_PROGRESS:
  save_progress(
      ROOT,
      "01a-coordinate-representations",
      {"objects": "green", "frames": "green", "rotation": "green"},
      exit_ticket_passed=exit_ticket_passed,
  )

## 学完请记住

关闭 Notebook 后应能脱稿说出：

1. 点是位置，自由向量是方向和长度；两者不是 NumPy 数组本身；
2. ${}^F p$ 表示同一几何点在 F 中的坐标；
3. 坐标系由原点和右手正交基组成；
4. ${}^A R_B$ 的列是 B 基轴在 A 中的表达；
5. 点转换包含平移，向量转换不包含平移；
6. 代码变量应明确 object type、frame、unit 和 shape。

<details>
<summary>展开参考答案（完成 Exit ticket 后再看）</summary>

- 点减点得到向量；
- 完整描述至少包含对象类型、表达坐标系和单位；
- 自由向量不随原点平移；
- `current_tip_pos + scaled_pos` 是点加向量。

</details>

## 反思与记录

在 `notes/01a-coordinate-representations.md` 写下：

1. 一个你过去把“点”和“数组”混为一谈的例子；
2. 用自己的话解释式 (3) 为什么有平移、式 (4) 为什么没有；
3. 从 Panda 源码抄录一处点、一处向量，并补充 frame、unit、shape 注释；
4. Exit ticket 中仍为黄色或红色的知识点。

下一本：[01B 坐标系与刚体变换](01b_frames_and_rigid_transforms.ipynb)。